### Data Splitting
The data proportion would be 70% training, 15% validation, 15% testing

In [1]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split


#### Configuration
---

In [2]:

ORIGINAL_DATASET = r"C:\Users\USER\Documents\Thesis Dataset\Original Dataset"

# Defining path variables for splitting the data
BASE_OUTPUT = r"C:\Users\USER\Documents\Thesis Dataset\Processed Dataset"
TRAIN_DIR = os.path.join(BASE_OUTPUT, "train")
VAL_DIR   = os.path.join(BASE_OUTPUT, "val")
TEST_DIR  = os.path.join(BASE_OUTPUT, "test")

# Splitting percentage or proportion
train_ratio = 0.70
val_ratio   = 0.15
test_ratio  = 0.15

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")

#### Clean and Create Directories 

Remove old processed data to avoid duplicates and create paths for training, validation and testing

In [3]:

# Deletes an already existed file
if os.path.exists(BASE_OUTPUT):
    shutil.rmtree(BASE_OUTPUT)

# Creates the directory for the path variables
for dir_path in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(dir_path, exist_ok=True)


#### SPLIT & COPY FUNCTION

In [4]:

def split_and_copy(class_images, train_path, val_path, test_path):
    """Split the list of image paths into train/val/test and copy once."""
    if len(class_images) < 3:
        print(f"Folder is Empty or does not have an image: {os.path.basename(train_path)}")
        return

    random.shuffle(class_images)

    # Splitting Data: Train vs Temp (Val+Test)
    train_imgs, temp_imgs = train_test_split(
        class_images, test_size=(1 - train_ratio), random_state=42
    )
    # Splitting the Temp: Validation and Test
    val_imgs, test_imgs = train_test_split(
        temp_imgs,
        test_size=(test_ratio / (test_ratio + val_ratio)),
        random_state=42,
    )

    # Copy each image
    for img in train_imgs:
        shutil.copy2(img, train_path)
    for img in val_imgs:
        shutil.copy2(img, val_path)
    for img in test_imgs:
        shutil.copy2(img, test_path)


#### Main loop
Loop for each plant to be splitted and copy to the processed folder

split and copy function is used

In [5]:
for disease_class in sorted(os.listdir(ORIGINAL_DATASET)):
    class_path = os.path.join(ORIGINAL_DATASET, disease_class)
    if not os.path.isdir(class_path):
        continue

    images = [
        os.path.join(class_path, f)
        for f in os.listdir(class_path)
        if f.lower().endswith(IMG_EXTENSIONS)
    ]

    train_out = os.path.join(TRAIN_DIR, disease_class)
    val_out = os.path.join(VAL_DIR, disease_class)
    test_out = os.path.join(TEST_DIR, disease_class)

    os.makedirs(train_out, exist_ok=True)
    os.makedirs(val_out, exist_ok=True)
    os.makedirs(test_out, exist_ok=True)

    split_and_copy(images, train_out, val_out, test_out)

    print(f"{disease_class}: {len(images)} images processed ✅")

print("🎉 Dataset successfully split into 70% Train, 15% Validation, 15% Test!")

Banana_Cordana: 442 images processed ✅
Banana_Healthy: 1108 images processed ✅
Banana_Panama Disease: 835 images processed ✅
Banana_Pestalotiopsis: 117 images processed ✅
Banana_Yellow and Black Sigatoka: 2597 images processed ✅
Coconut_Bud Root Dropping: 514 images processed ✅
Coconut_Bud Rot: 470 images processed ✅
Coconut_Gray Leaf Spot: 2135 images processed ✅
Coconut_Healthy: 123 images processed ✅
Coconut_Leaf Rot: 1673 images processed ✅
Coconut_WCLWD_DryingofLeaflets: 1078 images processed ✅
Coconut_WCLWD_Flaccidity: 1069 images processed ✅
Coconut_WCLWD_Yellowing: 1084 images processed ✅
Sugarcane_Healthy: 522 images processed ✅
Sugarcane_Mosaic: 462 images processed ✅
Sugarcane_RedRot: 518 images processed ✅
Sugarcane_Rust: 514 images processed ✅
Sugarcane_Yellow: 505 images processed ✅
🎉 Dataset successfully split into 70% Train, 15% Validation, 15% Test!


### Data Augmentation
We will manipulate the data using **horizontal flip**, **vertical flip**, **brighten**, **darken**, **rotate**, and **contrast**.

In [6]:
import os
import random
from pathlib import Path
from PIL import Image, ImageEnhance, ImageOps, ImageDraw
import numpy as np
import shutil

#### ⚙️ Configurationn

In [7]:
# Original training set folder directory
TRAIN_DIR = r"C:\Users\USER\Documents\Thesis Dataset\Processed Dataset\train"

# New folder for augmented images
AUGMENT_DIR = r"C:\Users\USER\Documents\Thesis Dataset\Processed Dataset\train_augmented"

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")

# --- AUGMENTATION CONTROLS ---
# Number of augmented copies to generate for *each* original image
AUGMENTATION_FACTOR = 7 

# Augmentation parameter ranges
ROTATE_ANGLE = 25
SHEAR_DEGREE = 15 # New parameter: Max shear angle in degrees
BRIGHTNESS_RANGE = (0.7, 1.4)
CONTRAST_RANGE = (0.7, 1.4)
GAMMA_RANGE = (0.8, 1.4)
NOISE_STD = 10

### 🧩 Data Augmentation Functions
Below are the different augmentation techniques used to diversify the training dataset:

| **Function** | **Description** | **Simulated Effect** |
|-------------|-----------------|------------------------|
| horizontal_flip ↔️ | Flips the image horizontally (left ↔ right). | Simulates images of the object's opposite side or view. |
| random_brightness ☀️ | Randomly adjusts brightness within a controlled range. | Variation in ambient light or exposure settings. |
| random_contrast 🎨 | Randomly adjusts the contrast. | Varying image clarity due to distance or camera settings. |
| gamma_correction 🔆 | Applies gamma adjustment (non-linear color shift). | Models the effects of shadows, glare, or uneven lighting. |
| add_noise 🔊 | Adds Gaussian noise to the image pixels. | Imperfections and variations in camera sensor data. |
| rotate 🔄 | Rotates the image by up to ±25 degrees. | Slight camera tilt or object orientation changes. |
| shear_transform 📏 | Applies a random horizontal or vertical shear (slant). | Simulates images taken from a slightly skewed angle or viewpoint. |



> 💡 **Tip:**  
> Applying these augmentations helps the model generalize better and handle variations in lighting, orientation, noise, and texture.


In [8]:

def horizontal_flip(img):
    return img.transpose(Image.FLIP_LEFT_RIGHT)

def random_brightness(img):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    factor = random.uniform(*BRIGHTNESS_RANGE)
    return ImageEnhance.Brightness(img).enhance(factor)

def random_contrast(img):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    factor = random.uniform(*CONTRAST_RANGE)
    return ImageEnhance.Contrast(img).enhance(factor)

def rotate(img):
    angle = random.uniform(-ROTATE_ANGLE, ROTATE_ANGLE)
    return img.rotate(angle, expand=True).resize(img.size)

def gamma_correction(img):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    gamma = random.uniform(*GAMMA_RANGE)
    inv_gamma = 1.0 / gamma
    lut = [pow(i / 255., inv_gamma) * 255. for i in range(256)] * 3
    return img.point(lut)

def add_noise(img):
    arr = np.array(img, dtype=np.int16)
    noise = np.random.normal(0, NOISE_STD, arr.shape).astype(np.int16)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def shear_transform(img):
    """
    Applies a random shear transformation (slanted distortion).
    """
    width, height = img.size
    
    # Randomly select shear direction (horizontal or vertical)
    if random.choice([True, False]):
        # Horizontal Shear (X-axis)
        shear_angle = random.uniform(-SHEAR_DEGREE, SHEAR_DEGREE)
        shear_factor = np.tan(np.radians(shear_angle))
        
        # Define the transformation matrix: [a, b, c, d, e, f]
        # X' = a*X + b*Y + c
        # Y' = d*X + e*Y + f
        # Horizontal Shear (Y remains unchanged, X shifts by Y * shear_factor)
        # a=1, b=shear_factor, c=0, d=0, e=1, f=0
        matrix = [1, shear_factor, 0, 0, 1, 0]
        
    else:
        # Vertical Shear (Y-axis)
        shear_angle = random.uniform(-SHEAR_DEGREE, SHEAR_DEGREE)
        shear_factor = np.tan(np.radians(shear_angle))
        
        # Vertical Shear (X remains unchanged, Y shifts by X * shear_factor)
        # a=1, b=0, c=0, d=shear_factor, e=1, f=0
        matrix = [1, 0, 0, shear_factor, 1, 0]

    # Use the appropriate resampling filter
    try:
        resample_filter = Image.Resampling.BICUBIC
    except AttributeError:
        resample_filter = Image.BICUBIC 
        
    # Shear transformation requires the AFFINE transform type
    return img.transform(
        img.size, 
        Image.AFFINE,
        matrix,
        resample=resample_filter
    ).resize(img.size) # Resize back to original dimensions

# ==========================================================
# LIST OF AUGMENTATIONS TO BE RANDOMLY SELECTED (Now 7 functions)
# ==========================================================
RANDOM_AUGMENTATIONS = [
    horizontal_flip,
    random_brightness,
    random_contrast,
    gamma_correction,
    add_noise,
    rotate,
    shear_transform, # NEW 7th technique
]

# Get the total number of available augmentations for the new random range
MAX_AUGMENTATIONS = len(RANDOM_AUGMENTATIONS) # This is now 7

#### Folders for the augmented data

In [9]:
print("⚙️ Setting up directories...")
# WARNING: This deletes the existing augmented folder!
if os.path.exists(AUGMENT_DIR):
    shutil.rmtree(AUGMENT_DIR) 
shutil.copytree(TRAIN_DIR, AUGMENT_DIR)


# --- Main Processing Loop ---
count_original = 0
count_augmented = 0

⚙️ Setting up directories...


#### The augmentation process loop

In [ ]:

count_original = 0
count_augmented = 0

print(f"✨ Starting augmentation process for {TRAIN_DIR}")
print(f"Goal: Generate {AUGMENTATION_FACTOR} random copies per image (1-7 augmentations per copy).")

for root, _, files in os.walk(AUGMENT_DIR):
    for file in files:
        if file.lower().endswith(IMG_EXTENSIONS):

            src = os.path.join(root, file)
            
            try:
                # Open image for augmentation
                img = Image.open(src).convert("RGB")
                count_original += 1 
                
                rel_path = os.path.relpath(root, AUGMENT_DIR)
                save_base = root
                base_name, ext = os.path.splitext(file)
                
                # Loop to generate N random copies
                for i in range(AUGMENTATION_FACTOR):
                    
                    # Start with a clean copy of the original image
                    aug_img = img.copy()
                    
                    # Determine number of augmentations (1 to 7)
                    num_augs = random.randint(1, MAX_AUGMENTATIONS) 
                    
                    # Randomly select and apply a subset of functions
                    applied_funcs = random.sample(RANDOM_AUGMENTATIONS, num_augs)
                    # Clean up function names for the filename
                    applied_names = "_".join([f.__name__.replace('random_', '').replace('_transform', '') for f in applied_funcs])

                    for aug_func in applied_funcs:
                        aug_img = aug_func(aug_img)

                    # Save the randomly augmented image with a unique name
                    save_path = os.path.join(save_base, f"{base_name}_aug{i+1}_{num_augs}augs_{applied_names}{ext}")
                    aug_img.save(save_path)
                    count_augmented += 1
                        
            except Exception as e:
                print(f"🛑 Error processing or saving image {src}: {e}")
                continue
                    
# --- Results Summary ---
total_images = count_original + count_augmented
expected_images = count_original * (AUGMENTATION_FACTOR + 1)

print("\n🍀 Data Augmentation Complete!")
print(f"Original Images (Copied): {count_original}")
print(f"Augmented Images Generated: {count_augmented}")
print(f"Total Images in '{os.path.basename(AUGMENT_DIR)}': {total_images} (Expected: {expected_images})")
print(f"📁 Saved to: {AUGMENT_DIR}")

✨ Starting augmentation process for C:\Users\USER\Documents\Thesis Dataset\Processed Dataset\train
Goal: Generate 7 random copies per image (1-7 augmentations per copy).


#### TESTING AND PRINTING IMAGES

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

TRAIN_AUGMENTED_DIR = AUGMENT_DIR
IMG_SIZE = (224, 224)

def show_samples_per_class(directory):
    for disease_class in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, disease_class)
        if not os.path.isdir(class_path):
            continue

        print(f"\n🌿 Class: {disease_class}")
        img_files = [
            f for f in os.listdir(class_path)
            if f.lower().endswith(IMG_EXTENSIONS)
        ]
        if not img_files:
            print("No images found. Skipping.")
            continue

        plt.figure(figsize=(4, 4))
        img_file = random.choice(img_files)
        img_path = os.path.join(class_path, img_file)

        img = tf.io.read_file(img_path)
        img = tf.image.decode_image(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.uint8)

        plt.imshow(img.numpy())
        plt.title(disease_class)
        plt.axis("off")
        plt.tight_layout()
        plt.show()

show_samples_per_class(TRAIN_DIR)